# Parte 2: Modelado Predictivo de Evasión de Clientes

## Objetivo General

Desarrollar modelos predictivos capaces de prever qué clientes tienen mayor probabilidad de cancelar sus servicios en Telecom X.

## Pipeline del Proyecto

1. Cargar y explorar datos limpios de Parte 1
2. Preparar datos para modelado
3. Análisis de correlación y selección de variables
4. Entrenar múltiples modelos de clasificación
5. Evaluar y comparar rendimiento
6. Interpretar resultados e identificar factores clave de evasión
7. Generar conclusiones estratégicas

## 1. Cargar y Explorar Datos Tratados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
df = pd.read_csv('../data/processed/telecom_cleaned.csv')

print("Información del Dataset")
print("=" * 80)
print(f"\nDimensiones: {df.shape[0]} registros × {df.shape[1]} columnas")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nPrimeras filas:\n{df.head()}")
print(f"\nEstadísticas descriptivas:\n{df.describe()}")
print(f"\nValores faltantes:\n{df.isnull().sum().sum()} valores nulos en total")

## 2. Eliminar Columnas Irrelevantes

In [ ]:
print("Columnas antes de eliminar:")
print(df.columns.tolist())

columns_to_drop = []

for col in df.columns:
    unique_vals = df[col].nunique()
    if unique_vals == len(df) or (unique_vals == 1000 and df[col].dtype == 'object'):
        columns_to_drop.append(col)
        print(f"\nColoque bajo sospecha: {col} con {unique_vals} valores únicos")

if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
    print(f"\nColumnas eliminadas: {columns_to_drop}")
else:
    print("\nNinguna columna fue eliminada (sin columnas irrelevantes detectadas)")

print(f"\nDimensiones después: {df.shape}")

## 3. Codificar Variables Categóricas

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Columnas categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"Columnas numéricas ({len(numerical_cols)}): {numerical_cols}")

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"\nDimensiones después del encoding: {df_encoded.shape}")
print(f"Nuevas columnas creadas: {df_encoded.shape[1] - df.shape[1]}")
print(f"\nPrimeras columnas: {df_encoded.columns[:10].tolist()}")

## 4. Análisis de Distribución de Evasión (Churn)

In [ ]:
churn_col = 'Churn'

if churn_col not in df_encoded.columns:
    print("Buscando columna de churn...")
    possible_names = [col for col in df_encoded.columns if 'churn' in col.lower() or 'evadido' in col.lower()]
    if possible_names:
        churn_col = possible_names[0]
        print(f"Columna de churn encontrada: {churn_col}")

churn_counts = df_encoded[churn_col].value_counts()
churn_proportions = df_encoded[churn_col].value_counts(normalize=True) * 100

print("Distribución de Churn (Cancelaciones)")
print("=" * 50)
print(f"\nValores absolutos:\n{churn_counts}")
print(f"\nProporción (%):\n{churn_proportions.round(2)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts.plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Distribución Absoluta de Churn', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Cantidad')
axes[0].set_xticklabels(['No Evadidos', 'Evadidos'], rotation=0)

churn_proportions.plot(kind='pie', ax=axes[1], autopct='%1.2f%%', colors=['green', 'red'])
axes[1].set_title('Proporción de Churn (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\nDesequilibrio de Clases: {churn_proportions[1]:.2f}% evadidos vs {churn_proportions[0]:.2f}% activos")

## 5. Análisis de Correlación

In [ ]:
correlation_matrix = df_encoded.corr()

top_corr_with_churn = correlation_matrix[churn_col].sort_values(ascending=False)

print("Top 10 variables con mayor correlación a Churn")
print("=" * 50)
print(f"\n{top_corr_with_churn.head(11)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, ax=axes[0], cbar_kws={'label': 'Correlación'})
axes[0].set_title('Matriz de Correlación Completa', fontsize=12, fontweight='bold')

top_corr_with_churn[1:11].plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Correlaciones con Churn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Correlación')

plt.tight_layout()
plt.show()

## 6. Análisis Dirigido de Variables Clave

In [ ]:
print("Identificando variables clave para análisis")
print("=" * 50)

numeric_features = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [col for col in numeric_features if col != churn_col]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

top_vars = top_corr_with_churn[1:5].index.tolist()

for idx, var in enumerate(top_vars):
    if var in numeric_features:
        df_encoded.boxplot(column=var, by=churn_col, ax=axes[idx])
        axes[idx].set_title(f'{var} vs Churn', fontweight='bold')
        axes[idx].set_xlabel('Churn')
        axes[idx].set_ylabel(var)

plt.suptitle('Análisis de Variables Numéricas vs Churn', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\nEstadísticas por grupo de Churn:")
print("=" * 50)
for var in top_vars:
    if var in numeric_features:
        print(f"\n{var}:")
        print(df_encoded.groupby(churn_col)[var].describe().round(2))

## 7. Preparar Datos para Modelado

In [ ]:
X = df_encoded.drop(columns=[churn_col])
y = df_encoded[churn_col]

print("Preparación de Datos")
print("=" * 50)
print(f"\nVariables independientes (X): {X.shape}")
print(f"Variable dependiente (y): {y.shape}")
print(f"\nDistribución de y:\n{y.value_counts()}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nDatos de entrenamiento: {X_train.shape[0]} registros")
print(f"Datos de prueba: {X_test.shape[0]} registros")
print(f"\nProporción Train/Test: {X_train.shape[0]/len(X):.1%} / {X_test.shape[0]/len(X):.1%}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nDatos estandarizados")
print(f"Media de X_train_scaled: {X_train_scaled.mean():.6f}")
print(f"Desviación estándar de X_train_scaled: {X_train_scaled.std():.6f}")

## 8. Entrenar Modelos de Clasificación

In [ ]:
models = {}

print("Entrenando Modelos de Clasificación")
print("=" * 70)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr_model
print("\n✓ Logistic Regression entrenada")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model
print("✓ Random Forest entrenada")

try:
    from xgboost import XGBClassifier
    xgb_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    print("✓ XGBoost entrenada")
except ImportError:
    print("⚠ XGBoost no está disponible")

print(f"\nModelos entrenados: {len(models)}")

## 9. Evaluar Rendimiento de Modelos

In [ ]:
results = []

print("Evaluación de Rendimiento")
print("=" * 70)

for model_name, model in models.items():
    print(f"\n{model_name}")
    print("-" * 70)
    
    if model_name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })

results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("Resumen Comparativo de Modelos")
print("=" * 70)
print(results_df.to_string(index=False))

## 10. Interpretación de Importancia de Variables

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(16, 5))

for idx, (model_name, model) in enumerate(models.items()):
    print(f"\n{model_name} - Top 15 Variables Importantes")
    print("=" * 50)
    
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feature_importance_df = pd.DataFrame({
            'Feature': X.columns,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        top_features = feature_importance_df.head(15)
        print(top_features.to_string(index=False))
        
        ax = axes[idx] if len(models) > 1 else axes
        top_features.plot(x='Feature', y='Importance', kind='barh', ax=ax, legend=False)
        ax.set_title(f'{model_name}\nImportancia de Variables', fontweight='bold')
        ax.set_xlabel('Importancia')
    else:
        ax = axes[idx] if len(models) > 1 else axes
        coefficients = pd.DataFrame({
            'Feature': X.columns,
            'Coefficient': model.coef_[0]
        }).sort_values('Coefficient', key=abs, ascending=False).head(15)
        
        print(coefficients.to_string(index=False))
        
        coefficients.plot(x='Feature', y='Coefficient', kind='barh', ax=ax, legend=False)
        ax.set_title(f'{model_name}\nCoeficientes', fontweight='bold')
        ax.set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

## 11. Conclusiones y Recomendaciones Estratégicas

In [ ]:
print("CONCLUSIONES Y RECOMENDACIONES ESTRATÉGICAS")
print("=" * 70)

best_model = results_df.loc[results_df['ROC-AUC'].idxmax()]
print(f"\nMejor Modelo: {best_model['Model']}")
print(f"ROC-AUC: {best_model['ROC-AUC']:.4f}")
print(f"Recall: {best_model['Recall']:.4f}")
print(f"Precision: {best_model['Precision']:.4f}")

print("\n" + "=" * 70)
print("PRINCIPALES FACTORES QUE INFLUYEN EN LA CANCELACIÓN:")
print("=" * 70)

if 'Random Forest' in models:
    rf_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': models['Random Forest'].feature_importances_
    }).sort_values('Importance', ascending=False).head(10)
    
    print("\nVariables Más Importantes (Random Forest):")
    for idx, row in rf_importance.iterrows():
        print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\n" + "=" * 70)
print("RECOMENDACIONES PARA REDUCIR EL CHURN:")
print("=" * 70)

recommendations = """
1. ENFOQUE PREDICTIVO
   - Implementar el modelo seleccionado en producción para identificar 
     clientes de alto riesgo en tiempo real.
   - Crear un scoring de riesgo de churn para cada cliente.

2. SEGMENTACIÓN Y RETENCIÓN
   - Dirigirse proactivamente a clientes con score de riesgo alto
   - Ofrecer incentivos personalizados antes de que se vayan

3. MONITOREO Y MEJORA CONTINUA
   - Monitorear el desempeño del modelo en datos nuevos
   - Reentrenar regularmente con nuevos datos de clientes
   - A/B testing de estrategias de retención

4. INTEGRACIÓN CON NEGOCIOS
   - Colaborar con el equipo de customer success
   - Establecer procesos de escalada para clientes de alto riesgo
   - Medir ROI de las acciones de retención
"""

print(recommendations)

print("\n" + "=" * 70)
print("FIN DEL ANÁLISIS")
print("=" * 70)